# Linear and Polynomial Regression

This notebook covers simple linear regression, multivariate evaluation, non-negative least squares, and polynomial regression. Every reported metric uses predictions from the model being evaluated.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


## 1. Simple linear regression

A straight line is fitted to a synthetic one-feature dataset. The plotted line uses a sorted grid, so connected points follow increasing feature values.


In [ ]:
X_simple, y_simple = make_regression(
    n_samples=120, n_features=1, noise=18, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y_simple, test_size=0.25, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
test_predictions = linear_model.predict(X_test)

print(f"Coefficient: {linear_model.coef_[0]:.3f}")
print(f"Intercept: {linear_model.intercept_:.3f}")
print(f"Training R^2: {linear_model.score(X_train, y_train):.3f}")
print(f"Test R^2: {r2_score(y_test, test_predictions):.3f}")

x_grid = np.linspace(X_simple.min(), X_simple.max(), 300).reshape(-1, 1)
plt.figure(figsize=(9, 5))
plt.scatter(X_train[:, 0], y_train, alpha=0.6, label="Training data")
plt.scatter(X_test[:, 0], y_test, alpha=0.6, label="Test data")
plt.plot(x_grid[:, 0], linear_model.predict(x_grid), color="tab:red", linewidth=2, label="Regression line")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.title("Simple Linear Regression")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 2. Multivariate diabetes regression

The obsolete Boston housing example is replaced with scikit-learn's supported diabetes dataset. Standardization is included for a consistent workflow, although ordinary least squares itself does not require scaling for prediction quality.


In [ ]:
diabetes = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

diabetes_model = make_pipeline(StandardScaler(), LinearRegression())
diabetes_model.fit(X_train, y_train)
diabetes_predictions = diabetes_model.predict(X_test)

print(f"Test MSE: {mean_squared_error(y_test, diabetes_predictions):.2f}")
print(f"Test R^2: {r2_score(y_test, diabetes_predictions):.3f}")


## 3. Ordinary vs. non-negative least squares

`positive=True` constrains every coefficient to be non-negative. Each R² value below is calculated from its own model's predictions; this corrects the stale-prediction error in the previous notebook.


In [ ]:
models = {
    "Ordinary least squares": LinearRegression(),
    "Non-negative least squares": LinearRegression(positive=True),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print(f"{name} test R^2: {r2_score(y_test, predictions):.3f}")


## 4. Polynomial regression

Polynomial features allow a linear estimator to represent a curved relationship. The degree is stated consistently and performance is measured on held-out data.


In [ ]:
X_bmi = diabetes.data[:, [2]]
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(
    X_bmi, y, test_size=0.2, random_state=42
)

degree = 3
polynomial_model = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    StandardScaler(),
    LinearRegression(),
)
polynomial_model.fit(X_train, y_train)
polynomial_predictions = polynomial_model.predict(X_test)
print(f"Degree-{degree} test R^2: {r2_score(y_test, polynomial_predictions):.3f}")

x_grid = np.linspace(X_bmi.min(), X_bmi.max(), 300).reshape(-1, 1)
plt.figure(figsize=(9, 5))
plt.scatter(X_train[:, 0], y_train, alpha=0.45, label="Training data")
plt.scatter(X_test[:, 0], y_test, alpha=0.65, label="Test data")
plt.plot(x_grid[:, 0], polynomial_model.predict(x_grid), color="tab:red", linewidth=2, label=f"Degree {degree} curve")
plt.xlabel("BMI (standardized feature)")
plt.ylabel("Disease progression")
plt.title("Polynomial Regression on the Diabetes Dataset")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Takeaways

- R² must be computed from the predictions of the model being evaluated.
- Plot regression functions on a sorted grid to avoid misleading zigzag lines.
- Higher polynomial degree increases flexibility and should be selected with validation rather than training fit alone.
